# Simplified Architecture Comparison: MAE Transformer vs LSTM

Side-by-side comparison for the thesis report.
All values from checkpoint configs and source code (no torch required).

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Rectangle
from pathlib import Path

mpl.rcParams.update({
    "font.family": "DejaVu Sans", "mathtext.fontset": "dejavusans",
    "figure.facecolor": "white", "savefig.facecolor": "white",
})
FIG_DIR = Path("../../analysis_outputs/figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Parameters from v27 wandb config.yaml
MAE = dict(
    d_model=384, enc_layers=8, dec_layers=2, heads=8,
    mask_ratio=0.5, temporal_patch=3, window=72,
    mlp_ratio=4, N=155, V=6, V_target=5, K=13,
    n_params=25.6,
)
MAE["d_ff"] = MAE["d_model"] * MAE["mlp_ratio"]
MAE["W_p"] = MAE["window"] // MAE["temporal_patch"]

# Parameters from run_lstm_cloud.sh + lstm_baseline.py
LSTM = dict(
    hidden=1024, num_layers=3, dropout=0.1,
    mask_feature=True, window=72,
    V=6, V_target=5, K=13,
    n_params=21.1,
)
LSTM["input_size"] = LSTM["V"] * (2 if LSTM["mask_feature"] else 1)
LSTM["head_out"] = LSTM["K"] * LSTM["V_target"]

print("Parameters loaded")

In [ ]:
# Colours
C_MAE = "#1F5F6B"
C_LSTM = "#7B5EA7"
C_BG1, C_BG2 = "#F0F4F7", "#F4F0F7"
C_TXT, C_SUB = "#222222", "#777777"
C_ENC, C_DEC = "#D35F5F", "#D4A76A"
C_SPA, C_FFN = "#5BA55B", "#7BAFD4"
C_EMB, C_OUT = "#E8A838", "#2C7A5E"
C_RNN, C_HEAD = "#9B59B6", "#8E44AD"

def rbox(ax, x, y, w, h, label="", fc="#FFF", ec="#333",
         lw=0.8, fs=8, tc="#222", bold=False, zorder=2):
    b = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.008",
                        fc=fc, ec=ec, lw=lw, zorder=zorder)
    ax.add_patch(b)
    if label:
        ax.text(x+w/2, y+h/2, label, ha="center", va="center",
                fontsize=fs, color=tc, fontweight="bold" if bold else "normal",
                zorder=zorder+1, linespacing=1.3)
    return b

def arr(ax, x0, y0, x1, y1, c="#555", lw=0.8):
    ax.annotate("", xy=(x1, y1), xytext=(x0, y0),
                arrowprops=dict(arrowstyle="-|>", color=c, lw=lw), zorder=4)

print("Helpers defined")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(18, 10), gridspec_kw={"height_ratios": [1, 0.65]})

for ax in axes:
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.02)
    ax.axis("off")
    ax.set_aspect("auto")

# ════════════════════════════════════════════════════════════════════
# TOP ROW: MAE Transformer — horizontal flow
# ════════════════════════════════════════════════════════════════════
ax = axes[0]
ax.add_patch(FancyBboxPatch((-0.02, -0.02), 1.04, 1.04,
             boxstyle="round,pad=0.01", fc=C_BG1, ec=C_MAE, lw=1.5, zorder=0))

# Title
ax.text(0.50, 0.96, "MAE Transformer (v27)", ha="center", fontsize=14,
        fontweight="bold", color=C_MAE)
ax.text(0.50, 0.91, f"≈ {MAE['n_params']:.1f} M parameters",
        ha="center", fontsize=9, color=C_SUB, fontstyle="italic")

# Layout: horizontal blocks
bh = 0.55   # block height
by = 0.15   # block bottom
mid = by + bh / 2

# 1) INPUT
ix, iw = 0.01, 0.10
rbox(ax, ix, by, iw, bh, fc="#FDEBD0", ec=C_EMB, lw=1.0)
ax.text(ix+iw/2, mid+0.15, "Input", ha="center", fontsize=10,
        fontweight="bold", color=C_TXT)
ax.text(ix+iw/2, mid+0.02, f"N={MAE['N']}\nW={MAE['window']}\nV={MAE['V']}",
        ha="center", fontsize=9, color=C_TXT, linespacing=1.4)
ax.text(ix+iw/2, mid-0.15, "all stations\njointly",
        ha="center", fontsize=7, color=C_SUB, linespacing=1.3)

arr(ax, ix+iw, mid, 0.125, mid, c=C_EMB, lw=1.0)

# 2) EMBEDDINGS
ex, ew = 0.135, 0.12
rbox(ax, ex, by, ew, bh, fc="#FFF8EC", ec=C_EMB, lw=1.0)
ax.text(ex+ew/2, mid+0.22, "Embeddings", ha="center", fontsize=10,
        fontweight="bold", color=C_TXT)
ax.text(ex+ew/2, mid+0.14, f"→ d = {MAE['d_model']}", ha="center",
        fontsize=8, color=C_TXT)

emb_names = ["value (MLP)", "position", "topo (MLP)", "time", "step index"]
emb_cols = [C_EMB, C_SPA, C_SPA, "#4A90C4", "#7B68AE"]
n_emb = len(emb_names)
eb_h = 0.052
eb_w = ew - 0.015
eb_gap = 0.008
total_emb_h = n_emb * eb_h + (n_emb - 1) * eb_gap
eb_y0 = mid + total_emb_h / 2 - eb_h + 0.10
for i, (nm, col) in enumerate(zip(emb_names, emb_cols)):
    ey = eb_y0 - i * (eb_h + eb_gap)
    rbox(ax, ex+0.007, ey, eb_w, eb_h, nm, fc=col, ec="#888",
         fs=6.5, tc="white", lw=0.4)

ax.text(ex+ew/2, mid-0.23, "Σ → LayerNorm",
        ha="center", fontsize=7.5, color=C_SUB)

arr(ax, ex+ew, mid, 0.27, mid, c="#AAA", lw=1.0)

# 3) MASKING + PATCH
mx, mw = 0.28, 0.095
ph = bh * 0.40
# Masking block
rbox(ax, mx, by + bh - ph - 0.01, mw, ph,
     "MAE\nMasking", fc="#FDEAEA", ec="#C0392B", lw=1.0,
     fs=9, tc="#8B0000", bold=True)
ax.text(mx+mw/2, by + bh - 0.05 - ph/2 + 0.02,
        f"r = {MAE['mask_ratio']}\nwhole-station",
        ha="center", fontsize=7, color="#A04040", linespacing=1.3)
# Patching block
rbox(ax, mx, by + 0.01, mw, ph,
     "Patch &\nMerge", fc="#EEF2F7", ec="#8899AA", lw=1.0,
     fs=9, tc="#555", bold=True)
ax.text(mx+mw/2, by + 0.01 + ph/2 - 0.06,
        f"P={MAE['temporal_patch']}: {MAE['window']}→{MAE['W_p']}",
        ha="center", fontsize=7, color=C_SUB)

# arrow between masking and patching
ax.annotate("", xy=(mx+mw/2, by + 0.01 + ph),
            xytext=(mx+mw/2, by + bh - ph - 0.01),
            arrowprops=dict(arrowstyle="-|>", color="#888", lw=0.8), zorder=4)

arr(ax, mx+mw, mid, 0.39, mid, c="#8899AA", lw=1.0)

# 4) ENCODER
encx, encw = 0.40, 0.19
rbox(ax, encx, by, encw, bh, fc="#E8EDF2", ec="#8899AA", lw=1.2)
ax.text(encx+encw/2, mid+0.23,
        f"Encoder  ×{MAE['enc_layers']}",
        ha="center", fontsize=11, fontweight="bold", color=C_TXT)

# Stacked encoder sub-blocks
ib_w = encw - 0.025
ib_h = 0.095
ib_x = encx + 0.012
ib_gap = 0.015

ib_y1 = mid + 0.065
ib_y2 = mid - 0.045
ib_y3 = mid - 0.155

rbox(ax, ib_x, ib_y1, ib_w, ib_h, "Temporal\nself-attention",
     fc=C_ENC, tc="white", fs=8)
rbox(ax, ib_x, ib_y2, ib_w, ib_h, "Spatial\nself-attention",
     fc=C_SPA, tc="white", fs=8)
rbox(ax, ib_x, ib_y3, ib_w, ib_h,
     f"FFN\n{MAE['d_model']}→{MAE['d_ff']}→{MAE['d_model']}",
     fc=C_FFN, tc="white", fs=8)

ax.annotate("", xy=(ib_x+ib_w/2, ib_y2+ib_h),
            xytext=(ib_x+ib_w/2, ib_y1),
            arrowprops=dict(arrowstyle="-|>", color="#555", lw=0.7), zorder=4)
ax.annotate("", xy=(ib_x+ib_w/2, ib_y3+ib_h),
            xytext=(ib_x+ib_w/2, ib_y2),
            arrowprops=dict(arrowstyle="-|>", color="#555", lw=0.7), zorder=4)

ax.text(encx+encw/2, by+0.015,
        f"h={MAE['heads']}  d={MAE['d_model']}  + res + LN + drop path",
        ha="center", fontsize=6, color=C_SUB)

# K,V arrow: encoder → decoder
arr(ax, encx+encw, mid, 0.63, mid, c=C_SPA, lw=1.8)
ax.text((encx+encw + 0.63)/2, mid+0.04, "K, V",
        ha="center", fontsize=9, fontweight="bold", color=C_SPA)

# 5) DECODER
decx, decw = 0.64, 0.17
rbox(ax, decx, by, decw, bh, fc="#F0E8E0", ec="#AA9988", lw=1.2)
ax.text(decx+decw/2, mid+0.23,
        f"Decoder  ×{MAE['dec_layers']}",
        ha="center", fontsize=11, fontweight="bold", color=C_TXT)

db_w = decw - 0.02
db_h = 0.09
db_x = decx + 0.01
db_gap = 0.015

db_y1 = mid + 0.07
db_y2 = mid - 0.035
db_y3 = mid - 0.14

rbox(ax, db_x, db_y1, db_w, db_h,
     f"Lead Queries\nN×K = {MAE['N']}×{MAE['K']}",
     fc="#7B68AE", tc="white", fs=7.5)
rbox(ax, db_x, db_y2, db_w, db_h,
     "Cross-attention\nQ←dec, KV←enc",
     fc=C_DEC, tc="white", fs=7.5)
rbox(ax, db_x, db_y3, db_w, db_h,
     "+ Persistence\nresidual head",
     fc="#E8F5E9", ec=C_OUT, tc=C_OUT, fs=7.5)

ax.annotate("", xy=(db_x+db_w/2, db_y2+db_h),
            xytext=(db_x+db_w/2, db_y1),
            arrowprops=dict(arrowstyle="-|>", color="#555", lw=0.7), zorder=4)
ax.annotate("", xy=(db_x+db_w/2, db_y3+db_h),
            xytext=(db_x+db_w/2, db_y2),
            arrowprops=dict(arrowstyle="-|>", color="#555", lw=0.7), zorder=4)

ax.text(decx+decw/2, by+0.015,
        f"h={MAE['heads']}  d={MAE['d_model']}",
        ha="center", fontsize=6, color=C_SUB)

# ── EMBEDDING → DECODER: shared embeddings feed decoder queries ──
# Curved arrow from embeddings box bottom to decoder queries, going below
emb_bottom_x = ex + ew/2
emb_bottom_y = by
dec_query_bottom_x = db_x + db_w/2
dec_query_y = db_y1

# Draw a curved path below the main flow
curve_y = by - 0.08
ax.annotate("",
    xy=(dec_query_bottom_x, dec_query_y),
    xytext=(emb_bottom_x, emb_bottom_y),
    arrowprops=dict(
        arrowstyle="-|>", color="#7B68AE", lw=1.2,
        connectionstyle="arc3,rad=-0.25",
        linestyle="dashed",
    ), zorder=4)
ax.text((emb_bottom_x + dec_query_bottom_x) / 2, curve_y + 0.01,
        "spatial + temporal + step embeddings → decoder queries",
        ha="center", fontsize=7, color="#7B68AE", fontstyle="italic",
        bbox=dict(fc="white", ec="none", alpha=0.85, pad=1.5))

# arrow decoder → output
arr(ax, decx+decw, mid, 0.84, mid, c=C_OUT, lw=1.5)

# 6) OUTPUT
ox, ow = 0.85, 0.13
rbox(ax, ox, by, ow, bh, fc="#E8F5E9", ec=C_OUT, lw=1.0)
ax.text(ox+ow/2, mid+0.12, "Output", ha="center", fontsize=10,
        fontweight="bold", color=C_OUT)
ax.text(ox+ow/2, mid-0.02,
        f"{MAE['K']} leads\n×\n{MAE['N']} stations\n×\n{MAE['V_target']} vars",
        ha="center", fontsize=8, color=C_OUT, linespacing=1.3)
ax.text(ox+ow/2, mid-0.22, "0–6 h",
        ha="center", fontsize=9, color=C_SUB, fontweight="bold")

# Training footer
ax.text(0.50, 0.02,
        "Huber loss (δ=1)   |   AdamW  lr=1e−4  wd=0.05   |   100 epochs   |   A100 GPU",
        ha="center", fontsize=7.5, color=C_SUB)


# ════════════════════════════════════════════════════════════════════
# BOTTOM ROW: LSTM Baseline — horizontal flow
# ════════════════════════════════════════════════════════════════════
ax = axes[1]
ax.add_patch(FancyBboxPatch((-0.02, -0.02), 1.04, 1.04,
             boxstyle="round,pad=0.01", fc=C_BG2, ec=C_LSTM, lw=1.5, zorder=0))

ax.text(0.50, 0.95, "LSTM Baseline (v1)", ha="center", fontsize=14,
        fontweight="bold", color=C_LSTM)
ax.text(0.50, 0.88, f"≈ {LSTM['n_params']:.1f} M parameters",
        ha="center", fontsize=9, color=C_SUB, fontstyle="italic")

bh2 = 0.52
by2 = 0.18
mid2 = by2 + bh2 / 2

# 1) INPUT
rbox(ax, 0.03, by2, 0.15, bh2, fc="#F0E0F5", ec=C_LSTM, lw=1.0)
ax.text(0.105, mid2+0.12, "Input", ha="center", fontsize=10,
        fontweight="bold", color=C_TXT)
ax.text(0.105, mid2+0.00,
        f"1 station\nW={LSTM['window']}\nV′={LSTM['input_size']}",
        ha="center", fontsize=9, color=C_TXT, linespacing=1.4)
ax.text(0.105, mid2-0.16, "6 vars +\n6 mask flags",
        ha="center", fontsize=7, color=C_SUB, linespacing=1.3)

arr(ax, 0.18, mid2, 0.20, mid2, c=C_LSTM, lw=1.0)

# 2) No pre-processing
rbox(ax, 0.21, by2, 0.17, bh2, fc="#F8F4FC", ec="#CCC", lw=0.8)
ax.text(0.295, mid2+0.08, "No embeddings", ha="center", fontsize=9,
        color="#BBB", fontstyle="italic")
ax.text(0.295, mid2-0.02, "No masking", ha="center", fontsize=9,
        color="#BBB", fontstyle="italic")
ax.text(0.295, mid2-0.12, "No spatial info", ha="center", fontsize=9,
        color="#BBB", fontstyle="italic")

arr(ax, 0.38, mid2, 0.40, mid2, c="#CCC", lw=1.0)

# 3) LSTM
lstm_x, lstm_w = 0.41, 0.24
rbox(ax, lstm_x, by2, lstm_w, bh2, fc="#F0E0F5", ec=C_LSTM, lw=1.2)
ax.text(lstm_x+lstm_w/2, mid2+0.20,
        f"Stacked LSTM ×{LSTM['num_layers']}",
        ha="center", fontsize=11, fontweight="bold", color=C_TXT)

lb_w = lstm_w - 0.03
lb_h = 0.14
lb_x = lstm_x + 0.015
lb_y = mid2 - lb_h/2

rbox(ax, lb_x, lb_y, lb_w, lb_h,
     f"LSTM Cell\nhidden = {LSTM['hidden']}",
     fc=C_RNN, tc="white", fs=9)

# Recurrence loop
ax.annotate("", xy=(lb_x+lb_w-0.01, lb_y+lb_h+0.015),
            xytext=(lb_x+0.01, lb_y+lb_h+0.015),
            arrowprops=dict(arrowstyle="-|>", color=C_RNN, lw=1.0,
                           connectionstyle="arc3,rad=-0.3"), zorder=4)
ax.text(lb_x+lb_w/2, lb_y+lb_h+0.05, "recurrence over t",
        ha="center", fontsize=7, color=C_RNN, fontstyle="italic")

ax.text(lstm_x+lstm_w/2, by2+0.02,
        f"dropout={LSTM['dropout']}   |   temporal only — no station interaction",
        ha="center", fontsize=6.5, color=C_SUB)

arr(ax, lstm_x+lstm_w, mid2, 0.67, mid2, c=C_LSTM, lw=1.5)

# 4) LINEAR HEAD
hx, hw = 0.68, 0.13
rbox(ax, hx, by2, hw, bh2, fc="#E0D0F0", ec=C_LSTM, lw=1.0)
ax.text(hx+hw/2, mid2+0.12, "Linear\nHead", ha="center", fontsize=10,
        fontweight="bold", color=C_TXT, linespacing=1.3)
ax.text(hx+hw/2, mid2-0.04,
        f"{LSTM['hidden']}→{LSTM['head_out']}",
        ha="center", fontsize=9, color=C_TXT)
ax.text(hx+hw/2, mid2-0.16, f"{LSTM['K']}×{LSTM['V_target']}",
        ha="center", fontsize=8, color=C_SUB)

arr(ax, hx+hw, mid2, 0.83, mid2, c=C_HEAD, lw=1.5)

# 5) OUTPUT
rbox(ax, 0.84, by2, 0.14, bh2, fc="#E8E0F5", ec=C_LSTM, lw=1.0)
ax.text(0.91, mid2+0.12, "Output", ha="center", fontsize=10,
        fontweight="bold", color=C_LSTM)
ax.text(0.91, mid2-0.02,
        f"{LSTM['K']} leads\n×\n1 station\n×\n{LSTM['V_target']} vars",
        ha="center", fontsize=8, color=C_LSTM, linespacing=1.3)
ax.text(0.91, mid2-0.20, "0–6 h",
        ha="center", fontsize=9, color=C_SUB, fontweight="bold")

# Training footer
ax.text(0.50, 0.06,
        "Huber loss (δ=1)   |   AdamW  lr=1e−3  wd=0.0   |   60 epochs   |   A100 GPU",
        ha="center", fontsize=7.5, color=C_SUB)

# ── Global title ──
fig.suptitle("Architecture Comparison", fontsize=16, fontweight="bold",
             y=1.01, color=C_TXT)

plt.tight_layout(pad=0.6, h_pad=1.2)

for ext in ["pdf", "svg", "png"]:
    kw = dict(bbox_inches="tight")
    if ext == "png":
        kw["dpi"] = 300
    fig.savefig(FIG_DIR / f"architecture_comparison.{ext}", **kw)

print(f"Saved → {FIG_DIR}/architecture_comparison.{{pdf,svg,png}}")
plt.show()
plt.close()

## Key Architectural Differences

| | MAE Transformer (v27) | LSTM Baseline (v1) |
|---|---|---|
| **Parameters** | \u2248 25.6 M | \u2248 21.1 M |
| **Input scope** | All 155 stations jointly | 1 station at a time |
| **Spatial modelling** | Factorised spatial attention | None (spatially blind) |
| **Temporal modelling** | Temporal self-attention on patches | LSTM recurrence (72 steps) |
| **Embeddings** | 5 learned (value, pos, topo, time, step) | Raw values + mask flags |
| **MAE masking** | 50% whole-station at training | No masking |
| **Decoder** | Cross-attention with lead queries | Linear head from last hidden |
| **Persistence prior** | Residual head (last obs) | None |
| **d / hidden** | d = 384 | hidden = 1024 |
| **Depth** | 8 enc + 2 dec layers | 3 LSTM layers |
| **Loss** | Huber (\u03b4=1) | Huber (\u03b4=1) |
| **Optimizer** | AdamW, lr=1e-4, wd=0.05 | AdamW, lr=1e-3, wd=0.0 |

All values from wandb config.yaml (Transformer) and run_lstm_cloud.sh (LSTM).